# 04 Model Training

Goal: train baseline candidate models using a leakage-safe preprocessing pipeline.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from data_preprocessing import TARGET_COLUMN
from evaluation import binary_classification_metrics
from model_training import candidate_models, make_pipeline, save_model, split_features_target

PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "model_ready_data.csv"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
df = pd.read_csv(PROCESSED_DATA_PATH)
split = split_features_target(df, target_column=TARGET_COLUMN, test_size=0.2, random_state=42)
split.X_train.shape, split.X_test.shape, split.y_train.mean(), split.y_test.mean()

In [ ]:
rows = []
fitted_models = {}

for model_name, model in candidate_models(random_state=42).items():
    pipe = make_pipeline(model, split.X_train)
    pipe.fit(split.X_train, split.y_train)
    y_proba = pipe.predict_proba(split.X_test)[:, 1]
    metrics = binary_classification_metrics(split.y_test, y_proba, threshold=0.5)
    metrics["model"] = model_name
    rows.append(metrics)
    fitted_models[model_name] = pipe

comparison = pd.DataFrame(rows).sort_values(["pr_auc", "roc_auc", "recall"], ascending=False)
comparison.to_csv(TABLES_DIR / "baseline_model_comparison.csv", index=False)
comparison

In [ ]:
best_model_name = comparison.iloc[0]["model"]
best_path = save_model(fitted_models[best_model_name], "final_model")
best_model_name, best_path